# **(ADD THE NOTEBOOK NAME HERE)**

## Objectives

* Write your notebook objective here, for example, "Fetch data from Kaggle and save as raw data", or "engineer features for modelling"

## Inputs

* Write down which data or information you need to run the notebook 

## Outputs

* Write here which files, code or artefacts you generate by the end of the notebook 

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/mohammedluqmanriaz/Desktop/Code Instatute/capstone_try_two/2025_capstone_project/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/mohammedluqmanriaz/Desktop/Code Instatute/capstone_try_two/2025_capstone_project'

# Section 1

Section 1 content

Import the data that is going to be cleaned 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('jupyter_notebooks/Data_set/transactions.csv')  # Make sure this path is correct


In [ ]:
# Updated function to create a data dictionary
def create_data_dictionary(df):
    descriptions = {
        'transaction_id': 'Unique identifier for each transaction',
        'customer_id': 'Unique identifier assigned to each customer (e.g., CUSTXXXXX)',
        'transaction_date': 'Date and time when the transaction occurred',
        'transaction_type': 'Type of transaction (e.g., UPI, Debit Card, Bill Payment)',
        'transaction_amount': 'Amount involved in the transaction'
    }

    dictionary_data = []
    for column in df.columns:
        sample_values = df[column].dropna().head(3).tolist()
        sample_str = ', '.join([str(x) for x in sample_values])
        
        dictionary_data.append({
            'Column': column,
            'Data Type': str(df[column].dtype),
            'Missing Values': df[column].isnull().sum(),
            'Missing %': round((df[column].isnull().sum() / len(df)) * 100, 2),
            'Unique Values': df[column].nunique(),
            'Sample Values': sample_str,
            'Description': descriptions.get(column, 'Custom/Engineered column - description needed')
        })
        
    return pd.DataFrame(dictionary_data)

# Generate and display the data dictionary
raw_data_dictionary = create_data_dictionary(df)
print(raw_data_dictionary)

# Optional: Save to file
# raw_data_dictionary.to_csv('data_dictionary.csv', index=False)

Create a dictionary for the data set

In [ ]:
# Check for duplicate customer IDs
duplicate_customer_ids = df[df.duplicated(subset='customer_id', keep=False)]
duplicate_customer_count = duplicate_customer_ids['customer_id'].nunique()
print(f"\nNumber of duplicate customer IDs: {duplicate_customer_count}")
print("Duplicate customer ID records:")
print(duplicate_customer_ids.sort_values(by='customer_id').head(10))  # Display first 10 for brevity

# Check for duplicate transaction IDs
duplicate_transaction_ids = df[df.duplicated(subset='transaction_id', keep=False)]
duplicate_transaction_count = duplicate_transaction_ids['transaction_id'].nunique()
print(f"\nNumber of duplicate transaction IDs: {duplicate_transaction_count}")
if duplicate_transaction_count > 0:
    print("Duplicate transaction ID records:")
    print(duplicate_transaction_ids.sort_values(by='transaction_id'))
else:
    print("No duplicate transaction IDs found.")

---

Show all duplicates within Transactions COL and Customer ID Col

In [ ]:
df.isnull().sum()

Decide:

Drop rows/columns

Fill with mean/median/mode/forward-fill (for time series)

Impute or flag as unknown


Check for Duplicates
You've already done this, but to summarize:

Check for duplicate rows entirely:

python

Edit
df.duplicated().sum()
Remove them if necessary:

python

df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

Validate Data Types

In [ ]:
df.dtypes

Convert date strings to datetime


In [ ]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

Check for Outliers in Numeric Columns
within the transaction amount

In [ ]:
sns.boxplot(x=df['transaction_amount'])
plt.show()

Check for Invalid or Inconsistent Entries
Negative amounts? (e.g., refunds or errors)

Unrecognized transaction types?

In [ ]:
df['transaction_type'].value_counts()
df[~df['customer_id'].str.startswith('CUST')]

Standardize Categorical Values
Make sure categories are consistent:

In [ ]:
df['transaction_type'] = df['transaction_type'].str.strip().str.title()

Check Date Ranges
Ensure transaction_date is within expected limits (e.g., not in the future):

In [ ]:
df[df['transaction_date'] > pd.Timestamp.today()]

Remove Unnecessary Columns
If there are columns that:

Contain only one value

Are irrelevant to your analysis

You can remove them:

In [ ]:
#df = df.drop(columns=['column_name'])

Handle Zero or Near-Zero Variance Columns
Columns with almost no variance may not be useful:

In [ ]:
df.nunique()

Save as a new file

In [ ]:
df.to_csv('jupyter_notebooks/Data_set/transactions_clean.csv', index=False)

# Section 2

Section 2 content

Adding Matplotlip graphs to the cleansed data in order to show:

📉 Transaction counts by type

📦 Transaction amount spread per type

📈 Daily transaction volume trend

In [ ]:
df_daily = df.groupby(df['transaction_date'].dt.date).size()
plt.figure(figsize=(12, 5))
df_daily.plot()
plt.title('Number of Transactions Per Day')
plt.xlabel('Date')
plt.ylabel('Number of Transactions')
plt.grid(True)
plt.tight_layout()
plt.show()

📊 Transaction amount distribution

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['transaction_amount'], bins=50, edgecolor='black')
plt.title('Distribution of Transaction Amounts')
plt.xlabel('Transaction Amount')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

📉 Transaction counts by type

In [ ]:
df_type_counts = df['transaction_type'].value_counts()
plt.figure(figsize=(8, 5))
df_type_counts.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Number of Transactions by Type')
plt.xlabel('Transaction Type')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

📦 Transaction amount spread per type

In [ ]:
plt.figure(figsize=(10, 6))
df.boxplot(column='transaction_amount', by='transaction_type', grid=False)
plt.title('Transaction Amounts by Type')
plt.suptitle('')
plt.xlabel('Transaction Type')
plt.ylabel('Transaction Amount')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Machine Learning:

Machine Learning Plan

- Goal for Classification: Predict transaction_type based on transaction_amount, customer_id, and transaction_date.

- Goal for Regression: Predict transaction_amount from other features.

Model Types:

- DecisionTreeClassifier & DecisionTreeRegressor

- RandomForestClassifier & RandomForestRegressor

- Graphs to Generate:

- Feature Importance Plot

- Decision Tree Visualization

- Predicted vs. Actual Scatter Plot (Regression)

- Confusion Matrix (Classification)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, r2_score
#import seaborn as sns

# Convert dates

In [ ]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['day_of_week'] = df['transaction_date'].dt.dayofweek
df['hour'] = df['transaction_date'].dt.hour

Encode categorical variables

In [ ]:
le_type = LabelEncoder()
df['transaction_type_encoded'] = le_type.fit_transform(df['transaction_type'])

# ---------------- CLASSIFICATION MODEL ----------------

In [ ]:
X_class = df[['transaction_amount', 'day_of_week', 'hour']]
y_class = df['transaction_type_encoded']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train_c, y_train_c)

# Confusion Matrix

In [ ]:
y_pred_c = clf.predict(X_test_c)
cm = confusion_matrix(y_test_c, y_pred_c)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_type.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title("Decision Tree Classifier - Confusion Matrix")
plt.show()


# Feature Importance (Classification)

In [ ]:
X_reg = df[['day_of_week', 'hour', 'transaction_type_encoded']]
y_reg = df['transaction_amount']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg = DecisionTreeRegressor(max_depth=5, random_state=42)
reg.fit(X_train_r, y_train_r)

# Predictions

In [ ]:
y_pred_r = reg.predict(X_test_r)

# Scatter Plot: Actual vs Predicted

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_r, y_pred_r, alpha=0.5)
plt.xlabel("Actual Transaction Amount")
plt.ylabel("Predicted Transaction Amount")
plt.title("Decision Tree Regressor - Actual vs Predicted")
plt.show()

# Feature Importance (Regression)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=reg.feature_importances_, y=X_reg.columns)
plt.title("Feature Importance - Regression")
plt.show()

# Metrics

In [ ]:
print(f"Regression RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.2f}")
print(f"Regression R²: {r2_score(y_test_r, y_pred_r):.2f}")

# ---------------- TREE VISUALIZATION ----------------

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(clf, feature_names=X_class.columns, class_names=le_type.classes_, filled=True)
plt.title("Decision Tree Classifier Structure")
plt.show()

What This Does
Classification: Predicts transaction_type

Shows Confusion Matrix

Shows Feature Importance

Shows Decision Tree Diagram

Regression: Predicts transaction_amount

Shows Actual vs. Predicted Scatter Plot

Shows Feature Importance

Prints RMSE & R² score


NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Conclusions and Next Steps


* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
